In [0]:
from pyspark.sql import functions as F

sdf_silver_season = spark.table("mlb.silver.statcast_clean")

#ステップ２：打撃チーム列を作成
sdf_with_team = sdf_silver_season.withColumn("batting_team",F.when(F.col("inning_topbot") == "Bot", F.col("home_team")).otherwise(F.col("away_team")))

#ステップ３：ドジャースの打撃のみに絞り込み
sdf_dodgers = sdf_with_team.filter(F.col("batting_team") == "LAD")

In [0]:
#ステップ４-a:週別のチーム全体トレンド
sdf_dodgers_weekly = (sdf_dodgers.filter(F.col("launch_speed").isNotNull())
.withColumn("week",F.date_trunc("week", F.col("game_date")))
.withColumn("is_hard_hit",(F.col("launch_speed") >= 95).cast("int"))
.groupBy("week")
.agg(
    F.count("*").alias("batted_ball_count"),
    F.avg("launch_speed").alias("avg_exit_velo"),
    F.avg("launch_angle").alias("avg_launch_angle"),
    F.round(F.avg("is_hard_hit")/ F.count("*") * 100, 1).alias("hard_hit_rate")
)
.orderBy("week")
)
    
(sdf_dodgers_weekly.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable("mlb.gold.dodgers_team_weekly"))

In [0]:
#ステップ４-b:選手別の集計（不調が特定選手に偏っているか確認用）
sdf_dodgers_by_player = (sdf_dodgers
    .filter(F.col("launch_speed").isNotNull())
    .groupBy("batter")
    .agg(
        F.count("*").alias("batted_ball_count"),
        F.avg("launch_speed").alias("avg_exit_velo"),
        F.avg("launch_angle").alias("avg_launch_angle")
    )
    .filter(F.col("batted_ball_count") >= 30)) #サンプル数フィルター

#既存のplayer_lookupと結合して選手名をつける
sdf_lookup = spark.table("mlb.gold.player_lookup")
sdf_dodgers_by_player_named = (sdf_dodgers_by_player
    .join(sdf_lookup,sdf_dodgers_by_player.batter == sdf_lookup.key_mlbam, "left")
    .select("full_name","batted_ball_count","avg_exit_velo","avg_launch_angle")
    .orderBy(F.col("batted_ball_count").desc())
    )

(sdf_dodgers_by_player_named.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("mlb.gold.dodgers_batter_summary"))

display(sdf_dodgers_by_player_named)

Databricks visualization. Run in Databricks to view.

In [0]:
# ステップ５：実際の得失点データ（pandasで取得、Sparkとは別扱いでOK)

%pip install pybaseball

from pybaseball import schedule_and_record

dodgers_schedule = schedule_and_record(2026,"LAD")
print(dodgers_schedule[["Date","R","RA","W/L"]].tail(20)) #直近20試合の得点（R)・失点（RA)

データの確認

In [0]:
# ② ドジャースの打撃データが正しく絞り込めているか
display(spark.sql("""
    SELECT batting_team, COUNT(*) AS n
    FROM (
        SELECT CASE WHEN inning_topbot = 'Bot' THEN home_team ELSE away_team END AS batting_team
        FROM mlb.silver.statcast_clean
    )
    GROUP BY batting_team
    ORDER BY n DESC
"""))

In [0]:
# ③ 特定の強打者1人だけで確認(例:Freeman)
from pybaseball import playerid_lookup
lookup = playerid_lookup("freeman", "freddie")
print(lookup[["key_mlbam", "name_first", "name_last"]])

In [0]:
# ①全体のサニティチェック
display(spark.sql("""
    SELECT COUNT(*) AS total_batted_balls, AVG(launch_speed) AS overall_avg_exit_velo
    FROM mlb.silver.statcast_clean
"""))

In [0]:
# ③ Freemanの数値を個別に確認
display(spark.sql("""
    SELECT full_name, batted_ball_count, avg_exit_velo, avg_launch_angle
    FROM mlb.gold.dodgers_batter_summary
    WHERE full_name LIKE '%Freeman%'
"""))

In [0]:
# 全体のデータ範囲(Bronzeそのままでも、Silverクレンジング後でも確認可能)
display(spark.sql("""
    SELECT MIN(game_date) AS earliest_date, 
           MAX(game_date) AS latest_date,
           COUNT(DISTINCT game_date) AS distinct_days
    FROM mlb.silver.statcast_clean
"""))

In [0]:
from pyspark.sql import functions as F

sdf_silver_season = spark.table("mlb.silver.statcast_clean")

#ステップ２：打撃チーム列を作成
sdf_with_team = sdf_silver_season.withColumn("batting_team",F.when(F.col("inning_topbot") == "Bot", F.col("home_team")).otherwise(F.col("away_team")))

#ステップ３：ドジャースの打撃のみに絞り込み
sdf_dodgers = sdf_with_team.filter(F.col("batting_team") == "LAD")
#ステップ４：不調前/不調後のフラグを付与（大谷選手分析と同じ境界線）
sdf_dodgers = sdf_dodgers.withColumn("period", F.when(F.col("game_date") < "2026-08-18", "不調前").otherwise("不調後"))



In [0]:
#チーム全体の比較
sdf_dodgers_period_summary = (sdf_dodgers.withColumn("is_hard_hit",(F.col("launch_speed") >= 95).cast("int")).groupBy("period").agg(
    F.count("*").alias("batted_ball_count"),
    F.avg("launch_speed").alias("avg_exit_velo"),
    F.avg("launch_angle").alias("avg_launch_angle"),
    F.round(F.avg("is_hard_hit")/ F.count("*") * 100, 1).alias("hard_hit_rate")
)
.orderBy("period"))

(sdf_dodgers_period_summary.write.format("delta").mode("overwrite").saveAsTable("mlb.gold.dodgers_period_summary"))

display(sdf_dodgers_period_summary)

In [0]:
#選手別の期間比較（誰が特に落ちているか）
sdf_dodgers_player_period = (sdf_dodgers.groupBy("batter","period")
.agg(
    F.count("*").alias("batted_ball_count"),
    F.avg("launch_speed").alias("avg_exit_velo")
)
.filter(F.col("batted_ball_count") >= 15)
.join(sdf_lookup,F.col("batter") == sdf_lookup.key_mlbam, "left").select("full_name","period","batted_ball_count","avg_exit_velo")
)

# ② 選手×期間の集計(pivot前)に、不調期間の行が存在するか
display(sdf_dodgers_player_period.groupBy("period").count())

#選手ごとに横並びで見たい時はpivotが便利
sdf_dodgers_player_pivot = (sdf_dodgers_player_period
    .groupBy("full_name")
    .pivot("period",["不調前","不調後"])
    .agg(F.first("avg_exit_velo"))
    .withColumn("velo_drop",F.col("不調前") - F.col("不調後"))
    .orderBy(F.col("velo_drop").desc()))

(sdf_dodgers_player_pivot.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("mlb.gold.dodgers_player_velo_drop"))

display(sdf_dodgers_player_pivot)


In [0]:
import pandas as pd
from pybaseball import schedule_and_record

dodgers_schedule = schedule_and_record(2026, "LAD")
dodgers_schedule["Date"] = pd.to_datetime(dodgers_schedule["Date"].str.replace(r'\s*\(\d+\)', '', regex=True) + ", 2026")

before = dodgers_schedule[dodgers_schedule["Date"] < "2026-08-18"]
during = dodgers_schedule[dodgers_schedule["Date"] >= "2026-08-18"]

print("不調前 平均得点:", before["R"].mean())
print("不調期間 平均得点:", during["R"].mean())

In [0]:
#投球チーム列（打撃チームの逆）を作成し、ドジャースの等級のみに絞り込み
sdf_dodgers_pitching = (sdf_silver_season
    .withColumn("pitching_team",F.when(F.col("inning_topbot") == "Bot",F.col("away_team"))
    .otherwise(F.col("home_team"))) #裏=ホーム攻撃なので、投げているのはビジターではなく実際は逆
    .filter(F.col("pitching_team") == "LAD")
)

BEFORE = "不調前"
AFTER = "不調後"

sdf_dodgers_pitching = (sdf_dodgers_pitching
    .withColumn("period",
    F.when(F.col("game_date") < "2026-08-18",BEFORE).otherwise(AFTER)))

In [0]:
#チーム投手陣期間比較（被打球の質）
sdf_dodgers_pitching_summary = (sdf_dodgers_pitching
    .filter(F.col("launch_speed").isNotNull())
    .withColumn("is_hard_hit",(F.col("launch_speed") >= 95).cast("int"))
    .groupBy("period")
    .agg(
        F.count("*").alias("batted_ball_against"),
        F.avg("launch_speed").alias("avg_exit_velo_against"),
        F.round(F.sum("is_hard_hit")/F.count("*")*100,1).alias("pct_hard_hit_against"),
        F.avg("release_speed").alias("avg_release_speed"),
        F.avg("release_spin_rate").alias("avg_spin_rate")
    )
)

(sdf_dodgers_pitching_summary.write.format("delta").mode("overwrite").saveAsTable("mlb.gold.dodgers_pitching_summary"))

display(sdf_dodgers_pitching_summary)

In [0]:
#投手別の期間比較(誰が崩れているか)
sdf_dodgers_pitcher_period = (sdf_dodgers_pitching
    .filter(F.col("launch_speed").isNotNull())
    .withColumn("is_hard_hit",(F.col("launch_speed") >= 95).cast("int"))
    .groupBy("pitcher","period")
    .agg(
        F.count("*").alias("batted_ball_against"),
        F.avg("launch_speed").alias("avg_exit_velo_against"),
        F.round(F.sum("is_hard_hit")/F.count("*")*100,1).alias("pct_hard_hit_against"),
        F.avg("release_speed").alias("avg_release_speed"),
        F.avg("release_spin_rate").alias("avg_spin_rate")
    )
    .filter(F.col("batted_ball_against") >= 10)
)

# 期間差の計算（pivotで不調前/不調後を列に展開）
sdf_velo_increase = (sdf_dodgers_pitcher_period
    .groupBy("pitcher")
    .pivot("period",[BEFORE, AFTER])
    .agg(F.first("avg_exit_velo_against"))
    .withColumn("velo_against_increase", F.col(AFTER) - F.col(BEFORE))  # プラス=被弾質が悪化
    .select("pitcher","velo_against_increase")
)

sdf_dodgers_pitcher_period = (sdf_dodgers_pitcher_period
    .join(sdf_lookup, F.col("pitcher") == sdf_lookup.key_mlbam, "left")
    .join(sdf_velo_increase, "pitcher", "left")
    .select("full_name","period","batted_ball_against","avg_exit_velo_against","pct_hard_hit_against","avg_release_speed","avg_spin_rate","velo_against_increase")
)

(sdf_dodgers_pitcher_period.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("mlb.gold.dodgers_pitcher_period"))

display(sdf_dodgers_pitcher_period)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

被Exit Velocityの変化が大きい投手(上位)
選手	不調前→不調後	変化幅	備考
edgardo henriquez	80.1 → 84.4 mph	+4.3(悪化)	Hard-hit率も+4.8pt悪化。サンプルは双方とも少なめ(250/56)
blake treinen	82.2 → 85.9 mph	+3.7(悪化)	ただしHard-hit率は-4.5pt改善と矛盾あり。afterのサンプルが42球と少なく、ブレの可能性が高い
brock stewart	79.4 → 80.2 mph	+0.8(やや悪化)	Hard-hit率も+4.7pt悪化
tyler glasnow	84.0 → 84.5 mph	+0.5(ほぼ横ばい)	Hard-hit率は+4.1pt悪化しており、球速自体はむしろ微増
一方、明確に改善している投手も多い
選手	不調前→不調後	変化幅
evan phillips	86.2 → 81.5 mph	-4.7(改善)
edwin díaz	85.0 → 79.4 mph	-5.6(改善、ただしafterはわずか20球)
emmet sheehan	83.1 → 80.2 mph	-2.9(改善)
kyle hurt	84.9 → 81.9 mph	-3.0(改善)
roki sasaki	84.3 → 81.6 mph	-2.6(改善、Hard-hit率も-8.5pt改善)
yoshinobu yamamoto	81.9 → 80.8 mph	-1.0(改善)

主力先発(山本・佐々木)やセットアッパー格(Phillips、Hurt)がむしろ良くなっている一方、崩れているのはhenriquez・treinen・stewartといったブルペンの一部の投手に限定されています。これは「チーム全体の投手崩壊」ではなく、「一部リリーバーの不調」レベルの話であり、チームの得点力低迷を説明する主因にはなりにくい規模です。

In [0]:
print("不調前 平均失点(RA):", before["RA"].mean())
print("不調後 平均失点(RA):", during["RA"].mean())

In [0]:
print("不調前 平均得点(R):", before["R"].mean())
print("不調後 平均得点(R):", during["R"].mean())

数字を整理すると
	不調前	不調後	差
平均得点(R)	4.98	4.83	-0.14(わずかな低下)
平均失点(RA)	3.85	3.19	-0.65(改善)
得失点差	+1.13	+1.64	+0.51(むしろ改善)

得点はわずかに下がっていますが、それ以上に失点が大きく減っているため、得失点差で見ると不調期間の方がむしろ良いという結果になっています。これはStatcastで見た「打撃の不振」「投手陣はほぼ横ばい」という傾向とも、方向性としては矛盾しません(得点は確かに少し落ちている)が、「チームが苦戦した時期」という前提そのものと数字が噛み合っていない状態です。

In [0]:
print("不調前 戦績:")
print(before["W/L"].value_counts())
print("\n不調後 戦績:")
print(during["W/L"].value_counts())


不調前:75勝51敗(勝率 .595)
不調後:25勝11敗(勝率 .694)
8/18を境にすると、むしろ「不調後」の方が明らかに勝率が高いという結果です。得失点差の改善(前回の結果)とも整合しています。つまり、「8/18から不調に入った」という前提そのものが誤りだったことになります。

In [0]:
import pandas as pd

dodgers_schedule = dodgers_schedule.sort_values("Date").reset_index(drop=True)
dodgers_schedule["is_win"] = dodgers_schedule["W/L"].str.startswith("W").fillna(False).astype(int)

# 窓幅を10/15/20試合など複数試して、どこが一番苦戦していたかを見る
for window in [10, 15, 20, 30]:
    dodgers_schedule[f"rolling_winrate_{window}"] = (
        dodgers_schedule["is_win"].rolling(window=window).mean()
    )

# 最も勝率が低かった期間を、窓幅ごとに確認
for window in [10, 15, 20, 30]:
    col = f"rolling_winrate_{window}"
    worst_idx = dodgers_schedule[col].idxmin()
    worst_row = dodgers_schedule.loc[worst_idx]
    print(f"窓幅{window}試合: 最も勝率が低かった区間の終了日={worst_row['Date']}, 勝率={worst_row[col]:.3f}")

In [0]:
window = 20
col = f"rolling_winrate_{window}"
worst_idx = dodgers_schedule[col].idxmin()

worst_stretch = dodgers_schedule.iloc[worst_idx - (window - 1) : worst_idx + 1]

print("不振期間:", worst_stretch["Date"].min(), "〜", worst_stretch["Date"].max())
print("勝敗:", worst_stretch["is_win"].sum(), "勝", window - worst_stretch["is_win"].sum(), "敗")

In [0]:
# 例:不振期間が7/26〜8/16だった場合
sdf_dodgers = sdf_dodgers.withColumn(
    "period",
    F.when(F.col("game_date") < "2026-07-26", "不振前")
     .when(F.col("game_date") <= "2026-08-16", "不振期間")
     .otherwise("不振後")
)

In [0]:
from pyspark.sql import functions as F

SLUMP_START = "2026-07-26"
SLUMP_END = "2026-08-16"

def add_period_column(df, date_col="game_date"):
    return df.withColumn(
        "period",
        F.when(F.col(date_col) < SLUMP_START, "不振前")
         .when(F.col(date_col) <= SLUMP_END, "不振期間")
         .otherwise("不振後")
    )

不振期間が7/26〜8/16だったと特定できたので再集計する

In [0]:
sdf_silver_season = spark.table("mlb.silver.statcast_clean")

sdf_dodgers_batting = (sdf_silver_season
    .withColumn(
        "batting_team",
        F.when(F.col("inning_topbot") == "Bot", F.col("home_team"))
         .otherwise(F.col("away_team"))
    )
    .filter(F.col("batting_team") == "LAD")
    .filter(F.col("launch_speed").isNotNull())
)
sdf_dodgers_batting = add_period_column(sdf_dodgers_batting)

sdf_dodgers_period_summary = (sdf_dodgers_batting
    .withColumn("is_hard_hit", (F.col("launch_speed") >= 95).cast("int"))
    .groupBy("period")
    .agg(
        F.count("*").alias("batted_ball_count"),
        F.avg("launch_speed").alias("avg_exit_velo"),
        F.avg("launch_angle").alias("avg_launch_angle"),
        F.round(F.sum("is_hard_hit") / F.count("*") * 100, 1).alias("hard_hit_rate")
    )
    .orderBy(F.when(F.col("period") == "不振前", 1)
              .when(F.col("period") == "不振期間", 2)
              .otherwise(3))
)

(sdf_dodgers_period_summary.write.format("delta").mode("overwrite")
    .saveAsTable("mlb.gold.dodgers_period_summary"))

display(sdf_dodgers_period_summary)

In [0]:
sdf_dodgers_pitching = (sdf_silver_season
    .withColumn(
        "pitching_team",
        F.when(F.col("inning_topbot") == "Bot", F.col("away_team"))
         .otherwise(F.col("home_team"))
    )
    .filter(F.col("pitching_team") == "LAD")
    .filter(F.col("launch_speed").isNotNull())
)
sdf_dodgers_pitching = add_period_column(sdf_dodgers_pitching)

sdf_dodgers_pitching_summary = (sdf_dodgers_pitching
    .withColumn("is_hard_hit", (F.col("launch_speed") >= 95).cast("int"))
    .groupBy("period")
    .agg(
        F.count("*").alias("batted_ball_against"),
        F.avg("launch_speed").alias("avg_exit_velo_against"),
        F.round(F.sum("is_hard_hit") / F.count("*") * 100, 1).alias("hard_hit_rate_against"),
        F.avg("release_speed").alias("avg_release_speed")
    )
    .orderBy(F.when(F.col("period") == "不振前", 1)
              .when(F.col("period") == "不振期間", 2)
              .otherwise(3))
)

(sdf_dodgers_pitching_summary.write.format("delta").mode("overwrite")
    .saveAsTable("mlb.gold.dodgers_pitching_period_summary"))

display(sdf_dodgers_pitching_summary)

In [0]:
before = dodgers_schedule[dodgers_schedule["Date"] < SLUMP_START]
during = dodgers_schedule[(dodgers_schedule["Date"] >= SLUMP_START) & (dodgers_schedule["Date"] <= SLUMP_END)]
after = dodgers_schedule[dodgers_schedule["Date"] > SLUMP_END]

for label, period_df in [("不振前", before), ("不振期間", during), ("不振後", after)]:
    win_count = period_df["W/L"].str.startswith("W").sum()
    total = len(period_df)
    print(f"{label}: 得点{period_df['R'].mean():.2f} / 失点{period_df['RA'].mean():.2f} / {win_count}勝{total-win_count}敗")

何が起きているか

打撃側:Exit Velocity・Hard-hit率は事実上横ばい(誤差レベルの変動)。投手側:被Hard-hit率が23.6%→24.5%とわずかに悪化していますが、これも劇的とは言えません。

それにもかかわらず、勝率は.638→.350へと激変し、得失点差も+1.52から-1.20へと大きく転落しています。「打球の質」というプロセス指標がほぼ動いていないのに、結果だけが大きく動いているということは、この20試合の不振が選手個々のパフォーマンス低下というより、"結果の振れ(バリアンス)"によるものだった可能性が高いことを示しています。

具体的に考えられる要因は以下の通りです。

接戦での弱さ:同じ強さの打球・投球内容でも、タイミング(クラッチヒット、併殺崩れ、守備の些細なミスなど)次第で勝敗は大きく変わります。特に1点差・2点差の接戦を落とし続けていた可能性があります
特定の試合でのブルペン崩壊:被Hard-hit率が多少上がっている(23.6%→24.5%)ことから、一部の試合で集中的に打たれ、大量失点した試合が平均値を押し上げている可能性があります(先ほど見つけたhenriquez・treinen・stewartの不調と整合的です)
単純にサンプルが20試合と少ない:20試合という短い区間では、実力通りの結果にならないブレが普通に起こり得ます

In [0]:
worst_stretch_games = dodgers_schedule[
    (dodgers_schedule["Date"] >= SLUMP_START) & (dodgers_schedule["Date"] <= SLUMP_END)
].copy()

worst_stretch_games["run_diff"] = worst_stretch_games["R"] - worst_stretch_games["RA"]

print(worst_stretch_games[["Date", "R", "RA", "run_diff", "W/L"]].to_string(index=False))

負け試合(13試合)の内訳
分類	試合数	該当日
僅差負け(1点差、サヨナラ含む)	5試合	7/28, 8/01, 8/05, 8/07(サヨナラ), 8/13
中差負け(2〜3点差)	2試合	8/09, 8/15
大差負け(4点差以上)	6試合	7/26(-5), 7/31(-5), 8/02(-4), 8/03(-5), 8/04(-4), 8/16(-4)

13敗のうちほぼ半数(6試合、46%)が4点差以上の大差負けです。これは「接戦でのクラッチな弱さ」というより、特定の試合で一方的に崩れていることを示しています。これは以前見つけた投手側の発見(henriquez・treinen・stewartといった一部リリーバーの不調)と、きれいに繋がる可能性が高いです。

In [0]:
blowout_dates = ["2026-07-26", "2026-07-31", "2026-08-02", "2026-08-03", "2026-08-04", "2026-08-16"]

troubled_pitchers = ["henriquez", "treinen", "stewart"]  # pybaseballのlast name表記に合わせて調整可

# pitcher列・game_date列・full_name結合済みのテーブルから、大差負け試合での登板を抽出
display(spark.sql(f"""
    SELECT full_name, game_date, release_speed, launch_speed
    FROM mlb.silver.statcast_clean s
    JOIN mlb.gold.player_lookup p ON s.pitcher = p.key_mlbam
    WHERE CAST(game_date AS STRING) IN ({','.join([f"'{d}'" for d in blowout_dates])})
      AND LOWER(full_name) LIKE '%henriquez%' OR LOWER(full_name) LIKE '%treinen%' OR LOWER(full_name) LIKE '%stewart%'
"""))

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT full_name, game_date,
       COUNT(*) AS pitch_count,
       AVG(launch_speed) AS avg_exit_velo_allowed
FROM mlb.silver.statcast_clean s
JOIN mlb.gold.player_lookup p ON s.pitcher = p.key_mlbam
WHERE CAST(game_date AS STRING) IN ('2026-07-26','2026-07-31','2026-08-02','2026-08-03','2026-08-04','2026-08-16')
  AND (LOWER(full_name) LIKE '%henriquez%' OR LOWER(full_name) LIKE '%treinen%' OR LOWER(full_name) LIKE '%stewart%')
GROUP BY full_name, game_date
ORDER BY game_date, full_name

現時点での分析ストーリー(まとめ)
チームの得失点差は7/26〜8/16の20試合で大きく悪化(+1.52 → -1.20)
打撃・投手とも「打球の質」を示すStatcast指標(Exit Velocity・Hard-hit率)はほぼ横ばい
→ 実力の持続的な低下というより、**特定の試合での大量失点(6試合が4点差以上の大敗)**と、**僅差の負け越し(5試合)**という"結果の振れ"が主因
大差負けした試合は、一部の不調投手(henriquez・treinen・stewartなど)の登板と重なっている可能性が高い